In [1]:

from pathlib import Path
import sys
import pandas as pd
import tensorflow as tf
import mlflow
import mlflow.tensorflow

PROJECT_ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

from src.models.vgg16_unet import build_vgg16_unet
from src.models.metrics import MeanIoUIgnoreVoid
from src.data.cityscapes_labels import GROUPS, G
from src.data.cityscapes_tfdata import make_cityscapes_ds
from src.models.unet import build_unet
from src.models.metrics import MeanIoUIgnoreVoid

print("Project root:", PROJECT_ROOT)
print("Groups:", GROUPS, "void_id:", G["void"])

2026-03-06 10:42:29.454224: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-06 10:42:29.454483: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 10:42:29.485236: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 10:42:30.494370: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Project root: /home/aurelien/formation_openclassrooms/projet_8/cityseg-project
Groups: ['flat', 'human', 'vehicle', 'construction', 'object', 'nature', 'sky', 'void'] void_id: 7


In [2]:
CSV_PATH = PROJECT_ROOT / "data/manifests/cityscapes_pairs.csv"
df = pd.read_csv(CSV_PATH)

ROOT = PROJECT_ROOT / "data"
df["img_abs"]  = df["image_path"].apply(lambda p: str(ROOT / p))
df["mask_abs"] = df["mask_path"].apply(lambda p: str(ROOT / p))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)

TARGET_HW = (256, 512)
BATCH = 4

ds_train = make_cityscapes_ds(
    train_df["img_abs"].tolist(),
    train_df["mask_abs"].tolist(),
    target_hw=TARGET_HW,
    batch_size=BATCH,
    training=True,
    cache=False,
)

ds_val = make_cityscapes_ds(
    val_df["img_abs"].tolist(),
    val_df["mask_abs"].tolist(),
    target_hw=TARGET_HW,
    batch_size=BATCH,
    training=False,
    cache=False,
)

E0000 00:00:1772790153.991419   17688 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1772790153.994574   17688 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [6]:
NUM_CLASSES = len(GROUPS)
VOID_ID = G["void"]

model = build_vgg16_unet(
    input_shape=(256, 512, 3),
    num_classes=NUM_CLASSES,
    freeze_encoder=True,
)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
miou = MeanIoUIgnoreVoid(num_classes=NUM_CLASSES, void_id=VOID_ID)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=[miou],
)

In [7]:
EXPERIMENT_NAME = "cityseg-unet"
mlflow.set_tracking_uri(f"file:{PROJECT_ROOT}/mlruns")
mlflow.set_experiment(EXPERIMENT_NAME)

EPOCHS = 8

# callbacks utiles
cb = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="models/vgg16_unet_best.keras",
        monitor="val_miou_no_void",
        mode="max",
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_miou_no_void",
        mode="max",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_miou_no_void",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

with mlflow.start_run(run_name="vgg16_unet_frozen_256x512"):
    mlflow.log_params({
        "model": "vgg16_unet",
        "target_h": 256,
        "target_w": 512,
        "batch": BATCH,
        "lr": 1e-3,
        "epochs": EPOCHS,
        "freeze_encoder": True,
        "void_ignored": True,
    })

    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=EPOCHS,
        callbacks=cb,
    )

    best_val = max(history.history.get("val_miou_no_void", [0.0]))
    mlflow.log_metric("best_val_miou_no_void", float(best_val))

    model.save("models/vgg16_unet_final.keras")
    mlflow.log_artifact("models/vgg16_unet_final.keras", artifact_path="model")


Epoch 1/8


743/743 ━━━━━━━━━━━━━━━━━━━━ 2254s 3s/step - loss: 0.5199 - miou_no_void: 0.4797 - val_loss: 0.3231 - val_miou_no_void: 0.5605 - learning_rate: 0.0010
Epoch 2/8


2026-03-06 11:21:20.423874: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 10485760 bytes after encountering the first element of size 10485760 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


743/743 ━━━━━━━━━━━━━━━━━━━━ 2267s 3s/step - loss: 0.3158 - miou_no_void: 0.5970 - val_loss: 0.2963 - val_miou_no_void: 0.6066 - learning_rate: 0.0010
Epoch 3/8


2026-03-06 11:59:07.190153: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 10485760 bytes after encountering the first element of size 10485760 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


...

743/743 ━━━━━━━━━━━━━━━━━━━━ 2262s 3s/step - loss: 0.2757 - miou_no_void: 0.6481 - val_loss: 0.2827 - val_miou_no_void: 0.6348 - learning_rate: 0.0010
Epoch 5/8
743/743 ━━━━━━━━━━━━━━━━━━━━ 2306s 3s/step - loss: 0.2632 - miou_no_void: 0.6600 - val_loss: 0.2753 - val_miou_no_void: 0.6572 - learning_rate: 0.0010
Epoch 6/8


2026-03-06 13:53:05.600579: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 10485760 bytes after encountering the first element of size 10485760 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


743/743 ━━━━━━━━━━━━━━━━━━━━ 2343s 3s/step - loss: 0.2584 - miou_no_void: 0.6658 - val_loss: 0.3217 - val_miou_no_void: 0.6271 - learning_rate: 0.0010
Epoch 7/8
743/743 ━━━━━━━━━━━━━━━━━━━━ 2379s 3s/step - loss: 0.2495 - miou_no_void: 0.6745 - val_loss: 0.2772 - val_miou_no_void: 0.6634 - learning_rate: 0.0010
Epoch 8/8


2026-03-06 15:11:47.418849: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 10485760 bytes after encountering the first element of size 10485760 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


743/743 ━━━━━━━━━━━━━━━━━━━━ 2472s 3s/step - loss: 0.2480 - miou_no_void: 0.6783 - val_loss: 0.2673 - val_miou_no_void: 0.6629 - learning_rate: 0.0010
